# MichiganCast Demo 04 — Imbalance, Stability, Export and Serving（不平衡 + 稳定性 + 导出 + 服务）

本 notebook 用于展示生产化闭环能力：

1. 不平衡学习策略实验（T23）。
2. 训练稳定性评估（T24）。
3. 模型导出与独立推理加载（T25）。
4. 推理服务与监控（T35/T36）。

## 0. 本章目标与结论

**目标：**
1. 比较 class weight / focal / threshold moving 等策略对正类指标的影响。
2. 验证同配置双跑稳定性（seed 固定 + 早停 + 学习率调度）。
3. 完成 TorchScript 导出，并用独立脚本推理验证。
4. 展示 API 推理监控的输入统计、预测分布和延迟指标。

**结论模板（执行后补充）：**
- 哪个不平衡策略对 Recall/PR-AUC 提升更明显。
- 稳定性双跑的最大波动是否在可接受范围。
- 导出模型是否可被独立加载，服务端响应与监控是否正常。

## 1. 输入与输出（路径与产物）

| 类型 | 路径 |
|---|---|
| 不平衡策略代码 | `src/train/imbalance.py` |
| 稳定性训练代码 | `src/train/train.py` |
| 导出脚本 | `src/train/export.py` |
| 独立推理脚本 | `src/serve/infer_torchscript.py` |
| 服务与监控 | `src/serve/app.py`, `src/serve/monitoring.py` |
| 不平衡结果 | `artifacts/reports/imbalance_strategy_comparison.*` |
| 稳定性报告 | `artifacts/reports/stability_train_report*.json` |
| 导出模型 | `artifacts/models/*.ts` |
| 推理结果 | `artifacts/models/inference_output*.json` |
| 监控日志 | `artifacts/logs/inference_monitoring.jsonl` |

In [ ]:
from pathlib import Path
import subprocess
import json
import pandas as pd

paths_to_check = [
    'src/train/imbalance.py',
    'src/train/train.py',
    'src/train/export.py',
    'src/serve/infer_torchscript.py',
    'src/serve/app.py',
    'src/serve/monitoring.py',
]

for p in paths_to_check:
    print(f'{p}:', 'OK' if Path(p).exists() else 'MISSING')

## 2. 方法与实现（调用 `src` 模块）

默认不执行（避免 notebook 阻塞或长时训练）。要真实执行请把 `DEMO_EXECUTE=True`。

In [ ]:
DEMO_EXECUTE = False

def run_or_print(cmd: str):
    print('\n$ ' + cmd)
    if not DEMO_EXECUTE:
        print('[skip] DEMO_EXECUTE=False, only showing command.')
        return None
    proc = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
    return proc.returncode

### 2.1 不平衡策略实验（T23）

**目的：** 对比 `baseline_bce / weighted_bce / focal / threshold_moving`。

In [ ]:
cmd_imbalance = (
    'scripts/run_in_pytorch_env.sh python -m src.train.imbalance '
    '--input data/interim/traverse_city_daytime_clean_v1.csv '
    '--output-dir artifacts/reports '
    '--epochs 20 '
    '--batch-size 256 '
    '--device cpu'
)
run_or_print(cmd_imbalance)

### 2.2 稳定性训练双跑（T24）

**目的：** 同配置运行两次，量化关键指标波动。

In [ ]:
cmd_stability = (
    'scripts/run_in_pytorch_env.sh python -m src.train.train '
    '--input-csv data/interim/traverse_city_daytime_clean_v1.csv '
    '--image-dir data/processed/images/lake_michigan_64_png '
    '--output-json artifacts/reports/stability_train_report_demo.json '
    '--checkpoint-prefix models/pytorch/michigancast_stability_demo '
    '--stability-runs 2 '
    '--acceptable-delta 0.03 '
    '--device auto'
)
run_or_print(cmd_stability)

### 2.3 导出 TorchScript（T25）

**目的：** 将训练好的模型导出为服务侧可直接加载格式。

In [ ]:
cmd_export = (
    'scripts/run_in_pytorch_env.sh python -m src.train.export '
    '--checkpoint-path models/pytorch/michigancast_multimodal_best.pth '
    '--output-path artifacts/models/michigancast_multimodal.ts '
    '--metadata-path artifacts/models/michigancast_multimodal_metadata.json '
    '--device cpu'
)
run_or_print(cmd_export)

### 2.4 独立推理加载验证（T25）

**目的：** 验证导出模型可以在训练脚本外独立推理。

In [ ]:
cmd_infer = (
    'scripts/run_in_pytorch_env.sh python -m src.serve.infer_torchscript '
    '--model-path artifacts/models/michigancast_multimodal.ts '
    '--output-json artifacts/models/inference_output_demo.json '
    '--device cpu'
)
run_or_print(cmd_infer)

### 2.5 服务启动与监控（T35/T36）

服务命令（建议在终端单独运行，不在 notebook 内阻塞）：

```bash
scripts/run_in_pytorch_env.sh python -m src.serve.app \
  --model-path artifacts/models/michigancast_multimodal.ts \
  --host 127.0.0.1 --port 8011 --device cpu
```

启动后可调用：
- `GET /health`
- `POST /predict`
- `GET /metrics/summary`

## 3. 结果解读（图表与指标）

In [ ]:
imb_csv = Path('artifacts/reports/imbalance_strategy_comparison.csv')
imb_json = Path('artifacts/reports/imbalance_strategy_comparison.json')

if imb_csv.exists():
    df_imb = pd.read_csv(imb_csv)
    display(df_imb.sort_values('recall_delta_vs_baseline', ascending=False))
else:
    print('imbalance comparison csv not found. Run section 2.1 first.')

if imb_json.exists():
    payload = json.loads(imb_json.read_text(encoding='utf-8'))
    print('significant improvement:', payload.get('has_significant_positive_metric_improvement'))

In [ ]:
stability_json_candidates = [
    Path('artifacts/reports/stability_train_report_demo.json'),
    Path('artifacts/reports/stability_train_report.json'),
    Path('artifacts/reports/stability_train_report_smoke.json'),
]

found = None
for p in stability_json_candidates:
    if p.exists():
        found = p
        break

if found is not None:
    report = json.loads(found.read_text(encoding='utf-8'))
    print('stability report:', found)
    print('is_stable:', report.get('is_stable'))
    print('max_delta:', report.get('max_delta'))
    print('acceptable_delta:', report.get('acceptable_delta'))
else:
    print('stability report not found. Run section 2.2 first.')

In [ ]:
export_meta = Path('artifacts/models/michigancast_multimodal_metadata.json')
infer_out = Path('artifacts/models/inference_output_demo.json')

if export_meta.exists():
    meta = json.loads(export_meta.read_text(encoding='utf-8'))
    print('torchscript path:', meta.get('torchscript_path'))
    print('trace max abs diff:', meta.get('trace_max_abs_diff'))
else:
    print('export metadata not found. Run section 2.3 first.')

if infer_out.exists():
    infer_payload = json.loads(infer_out.read_text(encoding='utf-8'))
    print('inference probability:', infer_payload.get('rain_probability'))
else:
    print('inference output not found. Run section 2.4 first.')

### 3.1 演示讲解要点（建议）

1. 不平衡策略对 Recall 提升是否以 Precision 下降为代价。
2. 稳定性评估的价值：为什么要记录 max_delta。
3. 导出 + 独立推理 + API 监控如何构成可交付闭环。

## 4. 工程反思与下一步

- 反思：不平衡策略应否按 horizon 分开调优。
- 风险：导出模型和在线服务输入 schema 漂移。
- 下一步：把 demo notebook 与 README 导航串联，形成完整对外展示路径。

## Appendix. 复现实验命令

```bash
scripts/run_in_pytorch_env.sh python -m src.train.imbalance --output-dir artifacts/reports
scripts/run_in_pytorch_env.sh python -m src.train.train --stability-runs 2 --output-json artifacts/reports/stability_train_report_demo.json
scripts/run_in_pytorch_env.sh python -m src.train.export --checkpoint-path models/pytorch/michigancast_multimodal_best.pth --output-path artifacts/models/michigancast_multimodal.ts
scripts/run_in_pytorch_env.sh python -m src.serve.infer_torchscript --model-path artifacts/models/michigancast_multimodal.ts --output-json artifacts/models/inference_output_demo.json
scripts/run_in_pytorch_env.sh python -m src.serve.app --model-path artifacts/models/michigancast_multimodal.ts --host 127.0.0.1 --port 8011
```